# Caso práctico con  Spark Streaming

En este noteboo se utilizarán los datos de Kaggle de fraud detection

### importar librerias e inicializar spark

In [1]:
import findspark
findspark.init()

### Importar librerias

In [2]:
#%load_ext nb_black
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark = SparkSession.builder.getOrCreate()

24/11/18 12:51:31 WARN Utils: Your hostname, Gretels-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.8.31 instead (on interface en0)
24/11/18 12:51:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/18 12:51:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/11/18 12:51:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/11/18 12:51:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


### Leer CSV con spark

In [3]:
df = spark.read.csv("data/fraud_detection.csv", 
                    header=True, 
                    inferSchema=True)

### Ver que nombres tienen las columnas

In [4]:
df.columns

['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud']

### Quitar las columnas que tienen de encabezado el nombre "isFraud" y "isFlaggedFraud

In [5]:
df = df.drop("isFraud", "isFlaggedFraud")

### Mostrar los primeros 2 renglores junto con el encabezado

In [6]:
df.show(2)

+----+-------+-------+-----------+-------------+--------------+-----------+--------------+--------------+
|step|   type| amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|
+----+-------+-------+-----------+-------------+--------------+-----------+--------------+--------------+
|   1|PAYMENT|9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|
|   1|PAYMENT|1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|
+----+-------+-------+-----------+-------------+--------------+-----------+--------------+--------------+
only showing top 2 rows



Step maps a unit of time in the real world. In this case 1 step is 1 hour of time. So we can assume for this example that we have another job that runs every hour and gets all the transactions in that time frame.

### agrupar por los mismos valores que hay en la columna "step" y contar cuantos hay en cada grupo

In [20]:
df.groupBy("step").count().show()

+----+-----+
|step|count|
+----+-----+
|  31|   12|
|  34|30904|
|  28|    4|
|  26|  440|
|  27|   41|
|  12|36153|
|  22|12635|
|   1| 2708|
|  13|37515|
|   6| 1660|
|  16|42471|
|   3|  552|
|  20|40625|
|  40|34270|
|   5|  665|
|  19|51352|
|  15|44609|
|  37|34000|
|   9|37628|
|  17|43361|
+----+-----+
only showing top 20 rows



We can therefore save the output of that job by filtering on each step and saving it to a separate file. 

In [8]:
%%time
"""steps = df.select("step").distinct().collect()
for step in steps[:]:
    _df = df.where(f"step = {step[0]}")
    #by adding coalesce(1) we save the dataframe to one file
    _df.coalesce(1).write.mode("append").option("header", "true").csv("data/fraud")"""

CPU times: user 2 µs, sys: 1 µs, total: 3 µs
Wall time: 3.1 µs


'steps = df.select("step").distinct().collect()\nfor step in steps[:]:\n    _df = df.where(f"step = {step[0]}")\n    #by adding coalesce(1) we save the dataframe to one file\n    _df.coalesce(1).write.mode("append").option("header", "true").csv("data/fraud")'

### Tomar los datos de la carpeta que se llama 'fraud'

In [9]:
!cd data/fraud 

### Leer el csv con spark

In [10]:
part = spark.read.csv(
    "data/fraud/part-00000-897a9dd3-832b-4e43-bcdc-c0009cfec4f0-c000.csv",
    header=True,
    inferSchema=True,
)

### agrupar por los mismos valores que hay en la columna "step" y contar cuantos hay en cada grupo

In [11]:
part.groupBy("step").count().show()

+----+-----+
|step|count|
+----+-----+
|  34|30904|
+----+-----+



Let’s create a streaming version of this input, we'll read each file one by one as if it was a stream.

In [12]:
dataSchema = part.schema

In [13]:
dataSchema

StructType([StructField('step', IntegerType(), True), StructField('type', StringType(), True), StructField('amount', DoubleType(), True), StructField('nameOrig', StringType(), True), StructField('oldbalanceOrg', DoubleType(), True), StructField('newbalanceOrig', DoubleType(), True), StructField('nameDest', StringType(), True), StructField('oldbalanceDest', DoubleType(), True), StructField('newbalanceDest', DoubleType(), True)])

*maxFilesPerTrigger* allows you to control how quickly Spark will read all of the files in the folder. 
In this example we're limiting the flow of the stream to one file per trigger.


In [14]:
streaming = (
    spark.readStream.schema(dataSchema)
    .option("maxFilesPerTrigger", 1)
    .csv("data/fraud/")
)

Let's set up a transformation.

The nameDest column is the recipient ID of the transaction.

In [15]:
dest_count = streaming.groupBy("nameDest").count().orderBy(F.desc("count"))

Now that we have our transformation, we need to specify an output sink for the results. For this example, we're going to write to a memory sink which keeps the results in memory.

We also need to define how Spark will output that data. In this example, we'll use the complete output mode (rewriting all of the keys along with their counts after every trigger).

In this example we won't include activityQuery.awaitTermination() because it is required only to prevent the driver process from terminating when the stream is active.

So in order to be able to run this locally in a notebook we won't include it.

In [16]:
activityQuery = (
    dest_count.writeStream.queryName("dest_counts")
    .format("memory")
    .outputMode("complete")
    .start()
)

# include this in production
# activityQuery.awaitTermination()

import time

for x in range(50):
    _df = spark.sql(
        "SELECT * FROM dest_counts WHERE nameDest != 'nameDest' AND count >= 2"
    )
    if _df.count() > 0:
        _df.show(10)
    time.sleep(0.5)

24/11/18 12:51:39 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/9r/_9jxjvn120vddx501h8lv2gr0000gn/T/temporary-d4171dd0-fea6-4e98-b174-3abe6456a9a6. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
24/11/18 12:51:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


+-----------+-----+
|   nameDest|count|
+-----------+-----+
| C797728811|    2|
|C1117969305|    2|
+-----------+-----+



+-----------+-----+
|   nameDest|count|
+-----------+-----+
| C232678930|    4|
|C1344555666|    3|
| C667558218|    3|
|C1707815182|    3|
|C1265000705|    3|
| C337326042|    3|
| C426875724|    3|
| C803987301|    3|
|C1039953024|    3|
|C1896059630|    3|
+-----------+-----+
only showing top 10 rows



+-----------+-----+
|   nameDest|count|
+-----------+-----+
| C232678930|    4|
|C1344555666|    3|
| C667558218|    3|
|C1707815182|    3|
|C1265000705|    3|
|C1699183015|    3|
| C810672355|    3|
|C1955367546|    3|
| C337326042|    3|
|C1857999018|    3|
+-----------+-----+
only showing top 10 rows



+-----------+-----+
|   nameDest|count|
+-----------+-----+
| C232678930|    4|
|C1344555666|    3|
| C667558218|    3|
|C1707815182|    3|
|C1265000705|    3|
|C1699183015|    3|
| C810672355|    3|
|C1955367546|    3|
| C337326042|    3|
|C1857999018|    3|
+-----------+-----+
only showing top 10 rows



+-----------+-----+
|   nameDest|count|
+-----------+-----+
| C232678930|    4|
|C1344555666|    3|
| C667558218|    3|
|C1707815182|    3|
|C1265000705|    3|
|C1699183015|    3|
| C810672355|    3|
|C1955367546|    3|
| C337326042|    3|
|C1857999018|    3|
+-----------+-----+
only showing top 10 rows



... (58 outputs left out)

24/11/18 12:53:57 WARN TaskSetManager: Stage 417 contains a task of very large size (1994 KiB). The maximum recommended task size is 1000 KiB.
24/11/18 12:53:58 WARN TaskSetManager: Stage 423 contains a task of very large size (1994 KiB). The maximum recommended task size is 1000 KiB.


+-----------+-----+
|   nameDest|count|
+-----------+-----+
|C1386732390|   10|
|C1797724563|    9|
| C386707279|    9|
| C680344850|    8|
|C1720199936|    8|
|C1807112061|    8|
| C779503478|    8|
| C445334352|    8|
|C2053478872|    8|
|C2049145129|    8|
+-----------+-----+
only showing top 10 rows



24/11/18 12:54:00 WARN TaskSetManager: Stage 426 contains a task of very large size (1994 KiB). The maximum recommended task size is 1000 KiB.
24/11/18 12:54:02 WARN TaskSetManager: Stage 432 contains a task of very large size (1994 KiB). The maximum recommended task size is 1000 KiB.


+-----------+-----+
|   nameDest|count|
+-----------+-----+
|C1386732390|   10|
|C1797724563|    9|
| C386707279|    9|
| C680344850|    8|
|C1720199936|    8|
|C1807112061|    8|
| C779503478|    8|
| C445334352|    8|
|C2053478872|    8|
|C2049145129|    8|
+-----------+-----+
only showing top 10 rows



Check if stream is active

In [17]:
spark.streams.active[0].isActive

True

In [18]:
activityQuery.status

{'message': 'Processing new data',
 'isDataAvailable': True,
 'isTriggerActive': True}

If we  want to turn off the stream we'll run activityQuery.stop() to reset the query for testing purposes.

In [19]:
activityQuery.stop()

24/11/18 12:54:02 WARN Shell: Interrupted while joining on: Thread[Thread-190112,5,main]
java.lang.InterruptedException
	at java.base/java.lang.Object.wait(Native Method)
	at java.base/java.lang.Thread.join(Thread.java:1300)
	at java.base/java.lang.Thread.join(Thread.java:1375)
	at org.apache.hadoop.util.Shell.joinThread(Shell.java:1042)
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1002)
	at org.apache.hadoop.util.Shell.run(Shell.java:900)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1212)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1306)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1288)
	at org.apache.hadoop.fs.FileUtil.readLink(FileUtil.java:212)
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileLinkStatusInternal(RawLocalFileSystem.java:1113)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1102)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatus(RawLocalF